# 05 – M4: fine-tune PhoBERT (W3-02, W3-03)

**Bắt buộc dùng GPU:** Runtime → Change runtime type → **T4 GPU** → Save.
Kết quả lưu vào Drive chung `NLI/results/`, checkpoint vào `NLI/checkpoints/` (khoảng 0,5 GB mỗi lần chạy).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results', 'checkpoints']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

## Kiểm tra GPU và dữ liệu đã tách từ

In [ ]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU, bật T4 GPU rồi chạy lại từ ô 0')
need = [f'data/processed/{f}_seg.jsonl' for f in
        ['xnli_train_100k', 'xnli_dev', 'xnli_test', 'vianli_train', 'vianli_dev', 'vianli_test', 'vinli_test']]
missing = [f for f in need if not os.path.exists(f)]
print('Thiếu file tách từ:', missing if missing else 'không, đủ cả 7 file')

Nếu thiếu file tách từ thì chạy ô dưới (cần Java, Colab có sẵn; khoảng 15–20 phút). Đủ rồi thì bỏ qua.

In [ ]:
!python scripts/segment_data.py

## W3-02: chạy thử 1.000 mẫu, 1 epoch (khoảng 3–5 phút)
Mục đích là bắt lỗi trước khi chạy thật. Kết quả chạy thử lưu riêng ở `results/smoke/`, không lẫn với kết quả chính.

In [ ]:
!python scripts/run_transformer.py --model M4 --scenario S2 --smoke

In [ ]:
import json, glob
for f in sorted(glob.glob('results/smoke/M4_S2_*.json')):
    r = json.load(open(f))
    c = r['config']
    print(f"{r['test_set']}: n={r['n']}, accuracy={r['accuracy']:.3f}, macro-F1={r['macro_f1']:.3f}")
print('thiết bị:', c['device'], '| fp16:', c['fp16'], '| thời gian huấn luyện:', round(c['train_seconds']), 'giây',
      '| số tham số:', c['n_params'])
print('dev theo epoch:', c['history'])

**Chạy thử đạt khi:** không có lỗi, `thiết bị: cuda`, `fp16: True`, có 2 dòng `vianli_test` và `vinli_test` (n=200).
Điểm thấp là bình thường, vì mới học 1.000 mẫu trong 1 epoch (khoảng 32 bước). Mô hình có thể đoán toàn một nhãn
(accuracy khoảng 0,33, macro-F1 khoảng 0,17).

---
## W3-03: PhoBERT kịch bản S2
So 2 cấu hình trên **dev**, mỗi cấu hình lưu riêng ở `results/tuning/`. Cấu hình có macro-F1 dev cao hơn được chép ra `results/`
(kết quả chính) và giữ checkpoint ở `checkpoints/M4_S2_seed42/`.
- A: lr 2e-5, batch 32, 3 epoch;
- B (cấu hình bài ViANLI): lr 1e-5, batch 16, 7 epoch, dropout lớp phân loại 0,4.

Khoảng 15–25 phút. Hướng dẫn: `docs/huong_dan/W3-03.md`.

In [ ]:
!python scripts/run_transformer.py --model M4 --scenario S2 --config A B

In [ ]:
import json
t = json.load(open('results/analysis/M4_tuning_S2_seed42.json'))
for r in t['results']:
    print(f"cấu hình {r['config']}: dev macro-F1 {r['dev_macro_f1']:.3f}, tốt nhất ở epoch {r['best_epoch']}, "
          f"{round(r['train_seconds'] / 60)} phút; dev theo epoch: {[h['dev_macro_f1'] for h in r['history']]}")
print('→ chọn cấu hình', t['chosen'])
for name in ['vianli_test', 'vinli_test']:
    r = json.load(open(f'results/M4_S2_{name}_seed42.json'))
    print(f"{name}: accuracy {r['accuracy']:.3f}, macro-F1 {r['macro_f1']:.3f}")